In [1]:
!pip install rpy2
%load_ext rpy2.ipython

In [2]:
%%R
print("R is working")

[1] "R is working"


In [3]:
# ============================================================
# NOTEBOOK: 2_sql_r_analysis.ipynb
# ============================================================

# ============================================================
# STEP 1 — INSTALL R SUPPORT
# ============================================================

!pip install rpy2

# ============================================================
# STEP 2 — LOAD R EXTENSION
# ============================================================

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [4]:
%%R

print("R is working successfully!")

[1] "R is working successfully!"


In [5]:
# ============================================================
# STEP 3 — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import zipfile
import os

# ============================================================
# STEP 4 — EXTRACT DATASET
# ============================================================

zip_path = "/content/northstar_dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/data")

print("DATASET EXTRACTED SUCCESSFULLY!")

# ============================================================
# STEP 5 — LOAD DATASETS
# ============================================================

customers = pd.read_csv('/content/data/northstar_dataset/customers.csv')
orders = pd.read_csv('/content/data/northstar_dataset/orders.csv')
deliveries = pd.read_csv('/content/data/northstar_dataset/deliveries.csv')
drivers = pd.read_csv('/content/data/northstar_dataset/drivers.csv')
vehicles = pd.read_csv('/content/data/northstar_dataset/vehicles.csv')
complaints = pd.read_csv('/content/data/northstar_dataset/complaints.csv')
incidents = pd.read_csv('/content/data/northstar_dataset/incidents.csv')
hubs = pd.read_csv('/content/data/northstar_dataset/hubs.csv')
app_events = pd.read_csv('/content/data/northstar_dataset/app_events.csv')

print("ALL DATASETS LOADED!")

DATASET EXTRACTED SUCCESSFULLY!
ALL DATASETS LOADED!


In [6]:
# ============================================================
# STEP 6 — SEND DATAFRAMES TO R
# ============================================================

%R -i customers
%R -i orders
%R -i deliveries
%R -i drivers
%R -i vehicles
%R -i complaints
%R -i incidents
%R -i hubs
%R -i app_events

print("DATA SENT TO R SUCCESSFULLY!")

DATA SENT TO R SUCCESSFULLY!


/usr/local/lib/python3.12/dist-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "order_id". Fall back to string conversion. The error is: Series can only be of one type, or None (and here we have <class 'float'> and <class 'str'>). If happening with a pandas DataFrame the method infer_objects() will normalize data types before conversion.
  warnings.warn('Error while trying to convert '


In [7]:
%%R

# ============================================================
# STEP 7 — INSTALL R PACKAGES
# ============================================================

install.packages("sqldf")
install.packages("dplyr")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/sqldf_0.4-12.tar.gz'
Content type 'application/x-gzip' length 61077 bytes (59 KB)
downloaded 59 KB


The downloaded source packages are in
	‘/tmp/RtmpV8GVSH/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/dplyr_1.2.1.tar.gz'
Content type 'application/x-gzip' length 923509 bytes (901 KB)
downloaded 901 KB


The downloaded source packages are in
	‘/tmp/RtmpV8GVSH/downloaded_packages’


In [8]:
%%R

# ============================================================
# STEP 8 — LOAD R LIBRARIES
# ============================================================

library(sqldf)
library(dplyr)

cat("R LIBRARIES LOADED SUCCESSFULLY!\n")

R LIBRARIES LOADED SUCCESSFULLY!


Loading required package: gsubfn
Loading required package: proto
Loading required package: RSQLite

Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union

In addition: Warning message:
no DISPLAY variable so Tk is not available 


In [9]:
%%R

# ============================================================
# QUERY 1 — FAILED DELIVERIES BY HUB
# ============================================================

query1 <- sqldf("
SELECT
    h.hub_name,
    COUNT(*) AS total_failed_deliveries
FROM deliveries d
JOIN hubs h
ON d.hub_id = h.hub_id
WHERE d.delivery_status = 'Failed'
GROUP BY h.hub_name
ORDER BY total_failed_deliveries DESC
")

print(query1)

        hub_name total_failed_deliveries
1  Midtown Relay                      26
2   Central Core                      23
3 North Exchange                      17
4      West Gate                      16
5    Airport Hub                      15
6  Riverside Hub                      14
7      East Dock                      11
8     South Link                      10


INTERPRETATION:

This query identifies operational hubs with the highest number of failed deliveries.
The findings may indicate inefficient dispatch operations, route planning problems,
or staffing and coordination issues within specific locations.

In [10]:
%%R

# ============================================================
# QUERY 2 — COMPLAINTS BY DELIVERY STATUS
# ============================================================

query2 <- sqldf("
SELECT
    d.delivery_status,
    COUNT(c.complaint_id) AS total_complaints
FROM deliveries d
JOIN orders o
ON d.order_id = o.order_id
JOIN complaints c
ON o.order_id = c.order_id
GROUP BY d.delivery_status
ORDER BY total_complaints DESC
")

print(query2)

  delivery_status total_complaints
1          OnTime              149
2         Delayed               48
3          Failed               35


INTERPRETATION:

This analysis investigates whether delayed or failed deliveries generate
higher complaint volumes and customer dissatisfaction.
The findings help identify operational factors affecting customer experience.

In [11]:
%%R

# ============================================================
# QUERY 3 — DRIVER PERFORMANCE ANALYSIS
# ============================================================

query3 <- sqldf("
SELECT
    dr.driver_id,
    ROUND(AVG(dr.driver_rating),2) AS avg_driver_rating,
    ROUND(AVG(dr.training_score),2) AS avg_training_score,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
    AS failed_deliveries
FROM drivers dr
JOIN deliveries d
ON dr.driver_id = d.driver_id
GROUP BY dr.driver_id
ORDER BY failed_deliveries DESC
LIMIT 15
")

print(query3)

   driver_id avg_driver_rating avg_training_score total_deliveries
1       D133              3.99               88.2               12
2       D104              3.45               87.7                7
3       D024              3.35               71.4                8
4       D131              4.26               86.7                9
5       D108              4.33               70.6               11
6       D092              4.24               88.2                5
7       D083              4.16               80.8                9
8       D055              5.00               90.5               10
9       D010              3.95               70.0                7
10      D004              4.75               88.9                9
11      D170              4.48               75.2                4
12      D165              3.89               82.2                6
13      D144              3.83               85.0                5
14      D143              4.14               68.5             

INTERPRETATION:

This query investigates whether lower driver ratings or weaker training scores
are associated with higher delivery failure rates.
The findings may reveal workforce capability and operational performance issues.

In [12]:
%%R

# ============================================================
# QUERY 4 — VEHICLE CONDITION VS INCIDENTS
# ============================================================

query4 <- sqldf("
SELECT
    v.vehicle_type,
    v.maintenance_status,
    COUNT(i.incident_id) AS total_incidents,
    ROUND(AVG(v.battery_health_pct),2) AS avg_battery_health
FROM vehicles v
JOIN deliveries d
ON v.vehicle_id = d.vehicle_id
JOIN incidents i
ON d.delivery_id = i.delivery_id
GROUP BY v.vehicle_type, v.maintenance_status
ORDER BY total_incidents DESC
")

print(query4)

   vehicle_type maintenance_status total_incidents avg_battery_health
1            EV             Active              61              82.46
2      CargoVan             Active              40              74.03
3        Hybrid             Active              30              69.92
4        Diesel           InRepair              25              71.35
5        Hybrid           InRepair              23              81.38
6            EV          Scheduled              22              86.24
7      CargoVan           InRepair              19              67.89
8        Diesel             Active              19              64.35
9            EV           InRepair              17              79.28
10       Hybrid          Scheduled              13              79.75
11     CargoVan          Scheduled               8              69.95
12       Diesel          Scheduled               3              75.87


INTERPRETATION:

This query evaluates whether vehicle condition and maintenance status
contribute to operational incidents and service reliability problems.
The findings support predictive maintenance analysis.

In [13]:
%%R

# ============================================================
# QUERY 5 — CUSTOMER DISSATISFACTION ANALYSIS
# ============================================================

query5 <- sqldf("
SELECT
    c.customer_id,
    cu.customer_type,
    COUNT(c.complaint_id) AS complaint_count,
    ROUND(AVG(c.compensation_amount),2) AS avg_compensation
FROM complaints c
JOIN customers cu
ON c.customer_id = cu.customer_id
GROUP BY c.customer_id, cu.customer_type
HAVING complaint_count >= 2
ORDER BY complaint_count DESC
LIMIT 20
")

print(query5)

   customer_id customer_type complaint_count avg_compensation
1        C0368      Consumer               4            19.38
2        C0110      Consumer               3            14.71
3        C0142      Consumer               3            29.57
4        C0172      Consumer               3            20.50
5        C0191      Consumer               3            20.92
6        C0242      Consumer               3            25.25
7        C0282      Consumer               3            24.88
8        C0372      Consumer               3            23.96
9        C0421      Consumer               3            39.66
10       C0545      Consumer               3            24.53
11       C0573           SME               3            37.14
12       C0626      Consumer               3            12.08
13       C0001           SME               2            21.95
14       C0004      Consumer               2            17.79
15       C0012      Consumer               2            17.33
16      

INTERPRETATION:

This query identifies customers experiencing repeated service failures.
Frequent complaints may indicate persistent operational inefficiencies,
high-risk delivery routes, or poor customer experience management.

In [14]:
%%R

# ============================================================
# QUERY 6 — ROUTE OVERRIDES VS DELIVERY STATUS
# ============================================================

query6 <- sqldf("
SELECT
    delivery_status,
    ROUND(AVG(manual_route_override_count),2)
    AS avg_route_overrides,
    COUNT(*) AS total_deliveries
FROM deliveries
GROUP BY delivery_status
ORDER BY avg_route_overrides DESC
")

print(query6)

  delivery_status avg_route_overrides total_deliveries
1         Delayed                1.07              202
2          Failed                1.04              132
3          OnTime                0.92              616


INTERPRETATION:

This query evaluates whether manual route overrides
are associated with delayed or failed deliveries.

The findings directly address one of the major operational concerns
identified in the case study.

In [15]:
%%R

# ============================================================
# QUERY 7 — HUB PERFORMANCE ANALYSIS
# ============================================================

query7 <- sqldf("
SELECT
    h.hub_name,
    h.capacity_score,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
    AS failed_deliveries,
    ROUND(AVG(d.customer_rating_post_delivery),2)
    AS avg_customer_rating
FROM hubs h
JOIN deliveries d
ON h.hub_id = d.hub_id
GROUP BY h.hub_name, h.capacity_score
ORDER BY failed_deliveries DESC
")

print(query7)

        hub_name capacity_score total_deliveries failed_deliveries
1  Midtown Relay             63              128                26
2   Central Core             88              115                23
3 North Exchange             82              136                17
4      West Gate             69              127                16
5    Airport Hub             71              104                15
6  Riverside Hub             66              115                14
7      East Dock             74              119                11
8     South Link             78              106                10
  avg_customer_rating
1                3.88
2                3.67
3                3.84
4                3.92
5                3.88
6                3.88
7                3.90
8                3.95


INTERPRETATION:

This query compares operational efficiency across hubs.
The findings help identify whether low-capacity hubs experience
higher failure rates and lower customer satisfaction.

In [16]:
%%R

cat("=====================================================\n")
cat("SQL IN R ANALYSIS COMPLETE\n")
cat("=====================================================\n")

cat("Completed Investigations:\n")
cat("1. Failed deliveries by hub\n")
cat("2. Complaints linked to delivery failures\n")
cat("3. Driver performance analysis\n")
cat("4. Vehicle maintenance and incidents\n")
cat("5. Customer dissatisfaction patterns\n")
cat("6. Route override behaviour\n")
cat("7. Hub efficiency comparison\n")

cat("\nNEXT NOTEBOOK:\n")
cat("3_r_analytics.ipynb\n")

SQL IN R ANALYSIS COMPLETE
Completed Investigations:
1. Failed deliveries by hub
2. Complaints linked to delivery failures
3. Driver performance analysis
4. Vehicle maintenance and incidents
5. Customer dissatisfaction patterns
6. Route override behaviour
7. Hub efficiency comparison

NEXT NOTEBOOK:
3_r_analytics.ipynb
